In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
packages = [
    "io.delta:delta-spark_2.12:3.0.0",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262"
]

In [3]:
spark = SparkSession.builder \
    .appName("gold_validation") \
    .master("local[*]") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/home/tan/safebank-data-platform/venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/tan/.ivy2/cache
The jars for the packages stored in: /home/tan/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b974e76e-0ac8-4ea0-aa59-0eb64e6905c3;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 196ms :: artifacts dl 12ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtim

In [4]:
df = spark.read.format("delta").load("s3a://gold/daily_financial_performance")
df.count()

508

In [5]:
df.printSchema()

root
 |-- report_date: date (nullable = true)
 |-- branch_city: string (nullable = true)
 |-- channel_code: string (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- total_amount_orginal: double (nullable = true)
 |-- total_amount_vnd: double (nullable = true)
 |-- processed_at: timestamp (nullable = true)



In [6]:
df.show(truncate=False)

+-----------+-----------+------------+-------------+------------------+--------------------+--------------------+--------------------------+
|report_date|branch_city|channel_code|currency_code|total_transactions|total_amount_orginal|total_amount_vnd    |processed_at              |
+-----------+-----------+------------+-------------+------------------+--------------------+--------------------+--------------------------+
|2025-12-03 |Dong Thap  |COUNTER     |VND          |8                 |2.0061693E8         |2.0061693E8         |2025-12-04 16:54:13.697268|
|2025-12-03 |Hung Yen   |APP         |USD          |31                |265823.72           |6.965810394969049E9 |2025-12-04 16:54:13.697268|
|2025-12-03 |Hai Phong  |WEB         |VND          |35                |1.203485765E9       |1.203485765E9       |2025-12-04 16:54:13.697268|
|2025-12-03 |Quang Ngai |WEB         |USD          |16                |236052.02000000002  |6.185654217273922E9 |2025-12-04 16:54:13.697268|
|2025-12-03 |

In [7]:
df = spark.read.format("delta").load("s3a://gold/loan_risk_snapshot")
df.count()

25/12/04 16:57:50 ERROR NonFateSharingFuture: Failed to get result from future
scala.runtime.NonLocalReturnControl


73

In [8]:
df.printSchema()

root
 |-- branch_city: string (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- status: string (nullable = true)
 |-- risk_category: string (nullable = true)
 |-- total_loans: long (nullable = true)
 |-- total_funded_amount: double (nullable = true)
 |-- total_outstanding_balance: double (nullable = true)
 |-- total_repaid_amount: double (nullable = true)
 |-- total_funded_amount_vnd: double (nullable = true)
 |-- total_outstanding_vnd: double (nullable = true)
 |-- total_repaid_amount_vnd: double (nullable = true)
 |-- avg_interest_rate: double (nullable = true)
 |-- snapshot_date: date (nullable = true)
 |-- processed_at: timestamp (nullable = true)



In [9]:
df.show(truncate=False)

+-----------+-------------+-------+-------------+-----------+-------------------+-------------------------+-------------------+-----------------------+---------------------+-----------------------+-----------------+-------------+--------------------------+
|branch_city|currency_code|status |risk_category|total_loans|total_funded_amount|total_outstanding_balance|total_repaid_amount|total_funded_amount_vnd|total_outstanding_vnd|total_repaid_amount_vnd|avg_interest_rate|snapshot_date|processed_at              |
+-----------+-------------+-------+-------------+-----------+-------------------+-------------------------+-------------------+-----------------------+---------------------+-----------------------+-----------------+-------------+--------------------------+
|Thai Nguyen|GBP          |ACTIVE |Standard     |1          |61914.53           |7943.15                  |53971.38           |61914.53               |7943.15              |53971.38               |5.59             |2025-12-04   |

In [10]:
df = spark.read.format("delta").load("s3a://gold/daily_security_summary")
df.count()

5479

In [11]:
df.printSchema()

root
 |-- report_date: date (nullable = true)
 |-- location_city: string (nullable = true)
 |-- device_model: string (nullable = true)
 |-- total_login: long (nullable = true)
 |-- total_failed_logins: long (nullable = true)
 |-- total_untrusted_device_logins: long (nullable = true)
 |-- total_strange_loc_logins: long (nullable = true)
 |-- processed_at: timestamp (nullable = true)



In [12]:
df.show(truncate=False)

+-----------+----------------+--------------+-----------+-------------------+-----------------------------+------------------------+--------------------------+
|report_date|location_city   |device_model  |total_login|total_failed_logins|total_untrusted_device_logins|total_strange_loc_logins|processed_at              |
+-----------+----------------+--------------+-----------+-------------------+-----------------------------+------------------------+--------------------------+
|2025-12-03 |South Ericberg  |Mac Studio    |1          |0                  |0                            |1                       |2025-12-04 16:55:35.763392|
|2025-12-03 |Christopherside |Pixel 10      |1          |0                  |0                            |1                       |2025-12-04 16:55:35.763392|
|2025-12-03 |Beasleyland     |Samsung Fold X|1          |0                  |0                            |1                       |2025-12-04 16:55:35.763392|
|2025-12-03 |Murphychester   |Iphone 17 

In [13]:
df = spark.read.format("delta").load("s3a://gold/merchant_consumption_analysis")
df.count()

25/12/04 16:57:53 ERROR NonFateSharingFuture: Failed to get result from future
scala.runtime.NonLocalReturnControl


837

In [14]:
df.printSchema()

root
 |-- report_date: date (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- total_spend_raw: double (nullable = true)
 |-- processed_at: timestamp (nullable = true)



In [15]:
df.show(truncate=False)

+-----------+-------------+---------------+----------------+-------+-------------------+------------------+---------------+--------------------------+
|report_date|merchant_name|category       |customer_city   |gender |age_group          |total_transactions|total_spend_raw|processed_at              |
+-----------+-------------+---------------+----------------+-------+-------------------+------------------+---------------+--------------------------+
|2025-11-30 |Viet Power   |Food & Beverage|West Shellyberg |Unknown|Millennials (25-40)|1                 |0.0            |2025-12-04 16:56:12.717508|
|2025-11-30 |Star Coffee  |Utilities      |Brittanymouth   |Female |Gen X (40-60)      |1                 |8.7443364E7    |2025-12-04 16:56:12.717508|
|2025-11-30 |Star Coffee  |Utilities      |Lake Georgeside |Female |Boomers (>60)      |1                 |2.9635218E7    |2025-12-04 16:56:12.717508|
|2025-11-30 |Global Taxi  |Transportation |East Angela     |Male   |Millennials (25-40)|1     

In [16]:
df = spark.read.format("delta") \
    .load("s3a://gold/customer_360")
df.count()

25/12/04 16:57:55 ERROR NonFateSharingFuture: Failed to get result from future
scala.runtime.NonLocalReturnControl


205

In [17]:
df.printSchema()

root
 |-- person_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthday: date (nullable = true)
 |-- city: string (nullable = true)
 |-- is_blocked: boolean (nullable = true)
 |-- total_balance_vnd: double (nullable = true)
 |-- num_accounts: long (nullable = true)
 |-- total_debt_vnd: double (nullable = true)
 |-- num_loans: long (nullable = true)
 |-- last_login_ts: timestamp (nullable = true)
 |-- login_count_lifetime: long (nullable = true)
 |-- age: integer (nullable = true)
 |-- net_worth: double (nullable = true)
 |-- segment: string (nullable = true)
 |-- report_date: date (nullable = true)
 |-- processed_at: timestamp (nullable = true)

